In [1]:
!pip install transformers[torch] accelerate -U -q

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings

warnings.simplefilter("ignore")

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [4]:
train['tweet'] = train['tweet'].fillna("")
test['tweet'] = test['tweet'].fillna("")

target_cols = [f's{i}' for i in range(1, 6)] + \
              [f'w{i}' for i in range(1, 5)] + \
              [f'k{i}' for i in range(1, 16)]

In [5]:
train_df, val_df = train_test_split(train, test_size=0.1, random_state=42)

In [6]:
model_name = "cardiffnlp/twitter-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=24,
    problem_type="regression"
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream 

In [9]:
class WeatherTweetDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len=128):
        self.texts = texts.values
        # Jeśli nie ma targetów (zbiór testowy), ustawiamy na None
        self.targets = targets.values if targets is not None else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        item_dict = {key: val.squeeze(0) for key, val in inputs.items()}

        if self.targets is not None:
            item_dict['labels'] = torch.tensor(self.targets[item], dtype=torch.float32)

        return item_dict

In [10]:
train_dataset = WeatherTweetDataset(train_df['tweet'], train_df[target_cols], tokenizer)
val_dataset = WeatherTweetDataset(val_df['tweet'], val_df[target_cols], tokenizer)
test_dataset = WeatherTweetDataset(test['tweet'], None, tokenizer)

In [11]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    rmse = np.sqrt(mean_squared_error(labels, predictions))
    return {"rmse": rmse}

In [12]:
training_args = TrainingArguments(
    output_dir='./roberta_results',
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="rmse",
    greater_is_better=False,
    report_to="none"
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

eval_results = trainer.evaluate()
print(f"Validation result (RMSE before post-processing): {eval_results['eval_rmse']:.5f}")

Epoch,Training Loss,Validation Loss,Rmse
1,0.020312,0.016224,0.127374
2,0.016803,0.015432,0.124225
3,0.014997,0.015378,0.124006


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Rmse
0.014997,0.015378,3,0.124006


Validation result (RMSE before post-processing): 0.12401


In [17]:

test_predictions = trainer.predict(test_dataset).predictions

In [18]:
# post processing
preds_post = np.clip(test_predictions, 0, 1)

s_sums = preds_post[:, 0:5].sum(axis=1, keepdims=True)
preds_post[:, 0:5] = preds_post[:, 0:5] / (s_sums + 1e-9)

w_sums = preds_post[:, 5:9].sum(axis=1, keepdims=True)
preds_post[:, 5:9] = preds_post[:, 5:9] / (w_sums + 1e-9)

In [19]:
submission_roberta = pd.DataFrame(preds_post, columns=target_cols)
submission_roberta.insert(0, 'id', test['id'])
submission_roberta.to_csv('submission_roberta.csv', index=False)